In [1]:
import asyncio
from agents import Agent, FileSearchTool, Runner, WebSearchTool
import os
from dotenv import load_dotenv

from IPython.display import display, Markdown
import nest_asyncio

import pandas as pd
from typing import Literal
from pydantic import BaseModel, Field
from sqlalchemy.util import await_only

from src.Agent_OCR import DocumentOCR, load_from_json
from src.Agent_User_Profile import *
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("personal_api_key")

## Prepare Data

In [3]:
w2_output =load_from_json("../Data/Intermediate/Client_Input_OCR/W2_XL_input_clean_2999.json")
complete_user_profile = str(dict({'Social_Security_Number': '522-86-4190', 'Name': 'Daniel Robinson', 'Email': 'Francisco_Reveriano@zoho.com', 'filingStatus': 'Single', 'dependents': 0, 'address': '8432 Castro Well West Anna NJ 73837-7432', 'state': 'NJ', 'Occupation': 'Consultant', 'spouse': SpouseProfile(name='', occupation='', birthYear=0), 'complete': True, 'complete_reasoning': 'The user supplied their email address, confirmed their filing status as Single, stated they have 0 dependents, and indicated their occupation is Consultant. Since these were the only missing pieces of information, and no spouse details are required for a Single filer, the profile is now complete.', 'missing_questions': []}))
complete_w2_profile = "TaxYear=2010 Employer='Hall Ltd Group' Employer_EIN='27-0313363' Employee_Name='Daniel Robinson' Employee_SSN='522-86-4190' Employee_DOB='' Employee_DoB='' Wages_Income=91282.31 Federal_Taxes_Withheld=31479.62 Filling_Status='Single' State_Information='State code SD, Employer state ID 951-27-584, State wages 96,230.09, State income tax 3,516.46, Locality Jeffery Burgs, Local wages 96,230.09, Local tax 16,429.64' Questions='Please provide your date of birth to complete the profile.' Score='Medium'"

In [4]:
user_information = '''
# Complete Employee Profile \n
{}

# Complete W-2 Profile \n
{}

# Employee W-2 \n
{}
'''.format(complete_user_profile, complete_w2_profile, w2_output)
print(len(user_information))

5642


## Agent

In [18]:
class Form1040Profile(BaseModel):
    Employee_SSN : str
    "Employee Social Security Number"
    TaxYear : int
    "Tax Year from Employee W-2"
    Employer: str
    "Employer Name from the Employee W-2"
    Employer_EIN: str
    "Employer Identification Number (EIN) from the Employee W-2"
    Employee_Name: str
    "Employee Name from the Employee W-2"
    Employee_DOB: str
    "Employee Date of Birth from the Employee W-2"
    Income: str
    ''' Total Employer Income'''
    Deduction: float
    '''What is the Total Employee Deductions?'''
    Deduction_Reasoning: str
    "What is the reasoning behind the Deductions? Give a detailed breakdown."
    taxCredits: float
    ''' What is the Total Tax Credits?'''
    taxCredits_Reasoning: str
    "What is the reasoning behind the Tax Credits? Give a detailed breakdown."
    taxWithheld: float
    '''What is the total tax withheld?'''
    taxWithheld_Reasoning: str
    "What is the reasoning behind the tax withheld? Give a detailed breakdown."
    federalTaxDue: float
    '''Federal tax due'''
    federalTaxDue_Reasoning: str
    "What is the reasoning behind the federal tax due? Give a detailed breakdown."
    refundAmount: float
    '''What is the Refund Amount (if positive)?'''
    refundAmount_Reasoning: str
    "What is the reasoning behind the Refund Amount? Give a detailed breakdown."
    amountOwed: float
    '''What is the Amount owed (if positive)?'''
    amountOwed_Reasoning: str
    "What is the reasoning behind the Amount owed? Give a detailed breakdown."
    questions: str
    "Any additional questions about the employee to accurately conduct calculations. Questions should be single item, detailed, and clear?"
    score: Literal["High", "Medium", "Low"]
    "Score for the user profile based on the questions and answers"

TaxAgent = Agent(
        name="TaxAgent",
        instructions="You are are preparing the 1040 tax document for an employee. Loop through and double-check everything.",
        output_type=Form1040Profile,
        model="o3",
        tools=[
            FileSearchTool(
                max_num_results=40,
                vector_store_ids=["vs_683dca6814108191a83ccb974fff986d"],
                include_search_results=True,
            )
        ],
    )

In [19]:
result = await Runner.run(
            TaxAgent, user_information
        )

print(result.final_output)

Employee_SSN='522-86-4190' TaxYear=2010 Employer='Hall Ltd Group' Employer_EIN='27-0313363' Employee_Name='Daniel Robinson' Employee_DOB='' Income='91282.31' Deduction=9350.0 Deduction_Reasoning='As a single filer in 2010 you are entitled to the standard deduction of $5,700 plus one personal exemption of $3,650, for a total deduction of $9,350.' taxCredits=0.0 taxCredits_Reasoning='No tax-credit information (education, child, retirement-saver, etc.) was provided, so zero credits are assumed for now.' taxWithheld=31479.62 taxWithheld_Reasoning='Box 2 of your 2010 W-2 shows $31,479.62 of federal income tax already withheld from your wages.' federalTaxDue=0.0 federalTaxDue_Reasoning='Your calculated regular income-tax liability on $81,932.31 of taxable income is about $16,664.33, which is fully covered by the $31,479.62 already withheld.' refundAmount=14815.29 refundAmount_Reasoning='Because withholding ($31,479.62) exceeds tax liability ($16,664.33) by $14,815.29, you are due a refund of